## Starter Notebook


In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

path = "BalancesApril2026\\BalancesApril2026\\AL-Energy-balance-sheets-April2026-edition.xlsb"   # your downloaded file
xl = pd.ExcelFile(path)
print(xl.sheet_names)

raw = xl.parse(xl.sheet_names[1], header=None)
raw.head(25)

['Cover', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000', '1999', '1998', '1997', '1996', '1995', '1994', '1993', '1992', '1991', '1990']


,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TOTAL,C0000X0350-0370,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
4,NaN,NaN,NaN,ktoe,NaN,2024.0,NaN,NaN,Total,Solid fossil fuels,...,Other liquid biofuels,Ambient heat (heat pumps),Non-renewable waste,Industrial waste (non-renewable),Non-renewable municipal waste,Nuclear heat,Heat,Electricity,Fossil energy,Bioenergy
5,+,Primary production,NaN,NaN,NaN,NaN,NaN,PPRD,1682.411,119.128,...,0,39.611,0,0,0,0,0,Z,781.484,135.623
6,+,Recovered & recycled products,NaN,NaN,NaN,NaN,NaN,RCV_RCY,0,0,...,Z,Z,Z,Z,Z,Z,Z,Z,0,0
7,+,Imports,NaN,NaN,NaN,NaN,NaN,IMP,1603.81,124.821,...,0,Z,0,0,0,Z,0,229.473,1237.859,136.478
8,-,Exports,NaN,NaN,NaN,NaN,NaN,EXP,990.979,71.429,...,0,Z,0,0,0,Z,0,200.763,788.907,1.308
9,+,Change in stock,NaN,NaN,NaN,NaN,NaN,STK_CHG,-37.014,0,...,0,Z,0,0,0,Z,Z,Z,-37.014,0


In [3]:
import pandas as pd, numpy as np


xl = pd.ExcelFile(path)

def parse_sheet(name):
    raw = xl.parse(name, header=None)
    country = raw.iloc[1, 0]
    unit    = raw.iloc[4, 3]
    year    = raw.iloc[4, 5]

    codes  = raw.iloc[3, 8:].values
    labels = raw.iloc[4, 8:].values

    body = raw.iloc[5:].copy()
    body = body[body[7].notna()]

    def flow_label(r):
        for c in (0, 1, 2):
            v = r[c]
            if isinstance(v, str) and v not in ("+", "-", "="):
                return v
        return None

    flow_names = body.apply(flow_label, axis=1)
    vals = body.iloc[:, 8:].copy()
    vals.columns = codes
    vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})
    vals = vals.apply(pd.to_numeric, errors="coerce")
    vals.insert(0, "flow", flow_names.values)
    vals.insert(0, "flow_code", body[7].values)

    out = vals.melt(id_vars=["flow_code", "flow"], var_name="siec", value_name="value")
    out["country"], out["unit"], out["year"] = country, unit, year
    return out

frames = []
for s in xl.sheet_names:
    try:
        f = parse_sheet(s)
        if pd.notna(f["country"].iloc[0]) and pd.notna(f["year"].iloc[0]):
            frames.append(f)
    except Exception as e:
        print("skip", s, "-", type(e).__name__, e)

df = pd.concat(frames, ignore_index=True)
df["year"] = df["year"].astype(int)
print(df.shape, df.country.nunique(), df.year.nunique())
df.head(10)

skip Cover - IndexError index 5 is out of bounds for axis 0 with size 4


C:\Users\marko\AppData\Local\Temp\ipykernel_20712\1362703973.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})
C:\Users\marko\AppData\Local\Temp\ipykernel_20712\1362703973.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})
C:\Users\marko\AppData\Local\Temp\ipykernel_20712\1362703973.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version

(347760, 7) 1 35


C:\Users\marko\AppData\Local\Temp\ipykernel_20712\1362703973.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})
C:\Users\marko\AppData\Local\Temp\ipykernel_20712\1362703973.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})


,flow_code,flow,siec,value,country,unit,year
0,PPRD,Primary production,TOTAL,1682.411,Albania,ktoe,2024
1,RCV_RCY,Recovered & recycled products,TOTAL,0.000,Albania,ktoe,2024
2,IMP,Imports,TOTAL,1603.810,Albania,ktoe,2024
3,EXP,Exports,TOTAL,990.979,Albania,ktoe,2024
4,STK_CHG,Change in stock,TOTAL,-37.014,Albania,ktoe,2024
5,GAE,Gross available energy,TOTAL,2258.228,Albania,ktoe,2024
6,INTMARB,International maritime bunkers,TOTAL,22.261,Albania,ktoe,2024
7,GIC,Gross inland consumption,TOTAL,2235.967,Albania,ktoe,2024
8,INTAVI,International aviation,TOTAL,87.904,Albania,ktoe,2024
9,NRGSUP,Total energy supply,TOTAL,2148.062,Albania,ktoe,2024


In [4]:
tot = df[df.siec == "TOTAL"]
w = tot.pivot_table(index=["country", "year"], columns="flow_code", values="value").reset_index()

w["import_dep"] = (w["IMP"] - w["EXP"]) / w["GIC"]
w["prod_share"] = w["PPRD"] / w["GIC"]
w[["GIC", "PPRD", "IMP", "EXP", "import_dep"]].describe()

flow_code,GIC,PPRD,IMP,EXP,import_dep
count,35.000000,35.000000,35.000000,35.000000,35.000000
mean,2014.896286,1476.589171,1050.742943,461.923086,0.277130
std,379.739189,398.354825,592.616237,436.560612,0.176409
min,1211.800000,882.998000,34.996000,5.933000,-0.034955
25%,1859.056000,1139.707000,561.283000,66.975000,0.141681
50%,2171.270000,1404.094000,1274.661000,244.107000,0.288968
75%,2248.003500,1767.934500,1496.352500,887.735500,0.436531
max,2672.987000,2459.993000,1920.474000,1246.949000,0.533600


In [ ]:
def parse_sheet_wide(name):
    raw = xl.parse(name, header=None)
    country, unit, year = raw.iloc[1, 0], raw.iloc[4, 3], raw.iloc[4, 5]

    codes = raw.iloc[3, 8:].tolist()
    prod_names = dict(zip(codes, raw.iloc[4, 8:].tolist()))

    body = raw.iloc[5:].copy()
    body = body[body[7].notna()].reset_index(drop=True)

    sign  = body[0].where(body[0].isin(["+", "-", "="]))
    sign  = sign.fillna(body[1].where(body[1].isin(["+", "-", "="])))
    level = body[[0, 1, 2]].apply(
        lambda r: next((i for i, v in enumerate(r) if isinstance(v, str) and v not in ("+", "-", "=")), None), axis=1)
    label = body[[0, 1, 2]].apply(
        lambda r: next((v for v in r if isinstance(v, str) and v not in ("+", "-", "=")), None), axis=1)

    vals = body.iloc[:, 8:].copy()
    vals.columns = codes
    vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan}).apply(pd.to_numeric, errors="coerce")

    meta = pd.DataFrame({"country": country, "year": year, "unit": unit,
                         "flow_code": body[7], "flow": label, "sign": sign, "level": level})
    return pd.concat([meta, vals], axis=1), prod_names

wide_frames, prod_names = [], {}
for s in xl.sheet_names:
    try:
        f, p = parse_sheet_wide(s)
        if pd.notna(f["country"].iloc[0]):
            wide_frames.append(f); prod_names.update(p)
    except Exception as e:
        print("skip", s, "-", e)

wide_all = pd.concat(wide_frames, ignore_index=True)
wide_all["year"] = wide_all["year"].astype(int)
#print(wide_all.shape)
wide_all

skip Cover - index 5 is out of bounds for axis 0 with size 4


C:\Users\marko\AppData\Local\Temp\ipykernel_20712\4108734840.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan}).apply(pd.to_numeric, errors="coerce")
C:\Users\marko\AppData\Local\Temp\ipykernel_20712\4108734840.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan}).apply(pd.to_numeric, errors="coerce")
C:\Users\marko\AppData\Local\Temp\ipykernel_20712\4108734840.py:20: FutureWarning: Downcasting

0       2024
1       2024
2       2024
3       2024
4       2024
        ... 
4825    1990
4826    1990
4827    1990
4828    1990
4829    1990
Name: year, Length: 4830, dtype: int64

In [6]:
print(len(xl.sheet_names), xl.sheet_names[:5], xl.sheet_names[-3:])

raw = xl.parse(xl.sheet_names[0], header=None)
print(raw.shape)

# rows where col 0 is text and the rest of the row is empty = likely country headers
hdr = raw[raw[0].apply(lambda v: isinstance(v, str)) & raw.iloc[:, 1:].isna().all(axis=1)]
print(hdr[0].tolist()[:50])
print(hdr.index.tolist()[:50])

36 ['Cover', '2024', '2023', '2022', '2021'] ['1992', '1991', '1990']
(13, 4)
['Energy balance sheets', 'This file presents energy balances sheets calculated with Eurostat methodology.', 'Albania', 'AL']
[5, 6, 11, 12]
